# Tutorial 18: Geo-Experiment Analysis with SyntheticDiD

A practitioner walkthrough for marketing analytics teams measuring lift from a campaign that ran in a subset of geographic markets. This tutorial leads with `SyntheticDiD` (Arkhangelsky et al. 2021) and validates against GeoLift's published Simulated Retail dataset in the closing section.

## 1. The Geo-Experiment Problem

Your team launched a marketing campaign in 18 of 80 metro markets last quarter. Sales data is available for all 80 markets, before and after launch. Did the campaign work, and by how much?

This is a textbook geo-experiment: a treatment ran in some markets and not others, you have weekly data spanning the launch, and leadership wants a single number with a confidence interval they can act on.

**Why this is hard.** Markets are heterogeneous - New York and Topeka are not interchangeable. Just averaging "treated minus control" misses that the treated markets might have been on a different trajectory anyway. With only 18 treated markets, cluster-robust standard errors are unstable. And parallel trends - the assumption underlying basic difference-in-differences - rarely holds across cities you didn't randomly assign.

**Why diff-diff.** The diff-diff library ships the canonical [Arkhangelsky et al. (2021)](https://www.aeaweb.org/articles?id=10.1257/aer.20190159) implementation of Synthetic Difference-in-Differences (SDiD). SDiD constructs a weighted blend of control markets whose pre-period trajectory matches the treated markets', plus a weighting of pre-periods that captures which baseline weeks are most informative. You get unit weights, time weights, and full panel-data interpretability - the version of synthetic control the AER paper introduced.

If you've been using GeoLift or CausalImpact, this is the SDiD walkthrough you were looking for in Python. We close the tutorial in Section 6 by running diff-diff against GeoLift's own published Simulated Retail dataset.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from diff_diff import (
    SyntheticDiD,
    generate_did_data,
    plot_synth_weights,
    practitioner_next_steps,
)

## 2. The Data

We'll use a synthetic panel that mirrors a typical marketing scenario:

- **80 markets**, named `market_000` through `market_079`
- **12 weekly periods**
- **18 treated markets** (campaign launched in week 7)
- **62 control markets**
- Outcome: weekly conversions per market (range roughly 700-3000)
- Pre-period: weeks 1-6  ·  Post-period: weeks 7-12
- True treatment effect: +300 conversions per treated market per week (about 18% lift on a baseline of ~1600)

We use diff-diff's `generate_did_data` and rescale the outcome to look like weekly marketing conversions. Because the data is synthetic, the true effect is known and we can verify SDiD recovers it.

In [ ]:
raw = generate_did_data(
    n_units=80,
    n_periods=12,
    treatment_period=6,        # periods 0-5 are pre, 6-11 are post (0-indexed)
    treatment_fraction=0.225,  # 18 of 80 markets
    treatment_effect=300.0,
    unit_fe_sd=400.0,
    time_trend=50.0,
    noise_sd=100.0,
    seed=42,
)

df = pd.DataFrame({
    "market_id": raw["unit"].apply(lambda u: f"market_{u:03d}"),
    "week": raw["period"] + 1,  # 1-indexed weeks 1..12
    "treated": raw["treated"],
    "conversions": (raw["outcome"] + 1500).round().astype(int),
})

print(f"Shape: {df.shape}")
n_treated = len(set(df.loc[df['treated'] == 1, 'market_id']))
print(f"Markets: {len(set(df['market_id']))} ({n_treated} treated, {len(set(df['market_id'])) - n_treated} control)")
print(f"Weeks: {df['week'].min()}-{df['week'].max()}  (pre: 1-6, post: 7-12)")

In [ ]:
df.head()

In [ ]:
# Mean conversions by treatment group and week (.round(0) for cleaner display)
df.groupby(["treated", "week"])["conversions"].mean().round(0).unstack()

In [ ]:
weeks = sorted(set(df["week"]))
treated_means = [df[(df.treated == 1) & (df.week == w)]["conversions"].mean() for w in weeks]
control_means = [df[(df.treated == 0) & (df.week == w)]["conversions"].mean() for w in weeks]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(weeks, treated_means, marker="o", label="Treated markets (n=18)", color="C0", linewidth=2)
ax.plot(weeks, control_means, marker="s", label="Control markets (n=62)", color="C1", linewidth=2)
ax.axvline(6.5, color="gray", linestyle="--", label="Campaign start")
ax.set_xlabel("Week")
ax.set_ylabel("Mean conversions per market")
ax.set_title("Pre-trends: treated vs control markets")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

The two lines are close but not identical in the pre-period (weeks 1-6). Treated markets are consistently a bit *below* control markets before the campaign launches. After week 7, treated jumps clearly above. This is exactly the situation SDiD is designed for: we can't pretend pre-trends are perfectly parallel, so we let the method find a weighted blend of control markets that matches the treated pre-trend more carefully.

## 3. SyntheticDiD - The Idea and the Fit

Synthetic Difference-in-Differences finds two sets of weights:

1. **Unit weights** ($\omega_j$): a weighted blend of control markets whose pre-period trajectory matches the treated markets' pre-period trajectory.
2. **Time weights** ($\lambda_t$): a weighting of pre-treatment periods that emphasizes the baseline weeks most informative for the comparison.

The ATT estimator combines both: take the time-weighted average of (treated mean minus unit-weighted control mean), then subtract the same quantity computed in the pre-period. The unit weights make the synthetic control match the treated group; the time weights make the comparison robust to pre-treatment level differences.

This is the method introduced in [Arkhangelsky, Athey, Hirshberg, Imbens, & Wager (2021)](https://www.aeaweb.org/articles?id=10.1257/aer.20190159), implemented in `diff_diff` to match the R `synthdid` reference (see `docs/methodology/REGISTRY.md` for the algorithmic details).

In [ ]:
# Note: variance_method, n_bootstrap, and seed are CONSTRUCTOR kwargs.
# They are not arguments to .fit().
model = SyntheticDiD(variance_method="placebo", n_bootstrap=100, seed=42)
results = model.fit(
    df,
    outcome="conversions",
    treatment="treated",
    unit="market_id",
    time="week",
    post_periods=[7, 8, 9, 10, 11, 12],
)

In [ ]:
print(results.summary())

**Plain-English interpretation.** SDiD estimates that the campaign lifted weekly conversions by **about 297 per treated market** (95% CI: 263 to 330), which is roughly an **18% lift** on a baseline of ~1600 weekly conversions per market. The true treatment effect in the synthetic DGP was 300, so SDiD recovered it within 1%.

The 95% CI is constructed from `results.conf_int` (note: the field name is `conf_int`, not `ci`), and the p-value crosses below the 0.01 significance threshold.

## 4. Diagnostics - Who Is the Synthetic Control?

SDiD's interpretability advantage: you can see exactly which control markets are doing the work. The unit weights tell you the weighted blend; the time weights tell you which baseline weeks the method emphasized; the pre-treatment fit RMSE tells you how well the synthetic match worked.

This is not a black box - you can show stakeholders the receipts.

In [ ]:
unit_weights = results.get_unit_weights_df()
print("Top 10 unit weights:")
print(unit_weights.head(10).to_string(index=False))
print(f"\nTotal weight: {unit_weights['weight'].sum():.4f}  (should be ~1.0)")
print(f"Markets with non-trivial weight (> 0.01): {(unit_weights['weight'] > 0.01).sum()}")

In [ ]:
# plot_synth_weights returns the Axes; in a notebook the plot displays inline
plot_synth_weights(results, weight_type="unit", top_n=10, min_weight=0.01)

In [ ]:
# Time weights: which pre-periods does SDiD lean on?
time_weights = results.get_time_weights_df()
print(time_weights.to_string(index=False))
print(f"\nSum: {time_weights['weight'].sum():.4f}  (should be 1.0)")

plot_synth_weights(results, weight_type="time")

In [ ]:
print(f"Pre-treatment fit RMSE: {results.pre_treatment_fit:.2f}")
print(f"This is well below the noise level (~137), so the synthetic control")
print(f"matches the treated pre-trend tightly. Good fit -> trustworthy effect.")

In [ ]:
# Treated mean vs synthetic counterfactual over time.
# We construct the synthetic by re-weighting control markets with the unit
# weights, then apply an intercept correction so the pre-period means match
# (this approximates the SDiD time-weight correction visually).
weight_lookup = dict(zip(unit_weights["unit"].tolist(), unit_weights["weight"].tolist()))

weeks = sorted(set(df["week"]))
treated_by_week = []
synth_raw = []
for w in weeks:
    week_rows = df[df["week"] == w]
    treated_by_week.append(float(week_rows[week_rows.treated == 1]["conversions"].mean()))
    control_rows = week_rows[week_rows.treated == 0]
    synth_raw.append(sum(
        float(c) * weight_lookup.get(m, 0.0)
        for m, c in zip(control_rows["market_id"], control_rows["conversions"])
    ))

# Intercept correction so pre-period means match
pre_idx = [i for i, w in enumerate(weeks) if w <= 6]
shift = (sum(treated_by_week[i] for i in pre_idx) - sum(synth_raw[i] for i in pre_idx)) / len(pre_idx)
synth_by_week = [s + shift for s in synth_raw]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(weeks, treated_by_week, marker="o", label="Treated markets (mean)", color="C0", linewidth=2)
ax.plot(weeks, synth_by_week, marker="s", label="Synthetic control", color="C1", linewidth=2, linestyle="--")
ax.axvline(6.5, color="gray", linestyle=":", label="Campaign start")
post_idx = [i for i, w in enumerate(weeks) if w >= 7]
ax.fill_between(
    [weeks[i] for i in post_idx],
    [treated_by_week[i] for i in post_idx],
    [synth_by_week[i] for i in post_idx],
    alpha=0.2, color="C0", label="Treatment effect",
)
ax.set_xlabel("Week")
ax.set_ylabel("Conversions")
ax.set_title("Treated markets vs synthetic control")
ax.legend()
plt.show()

## 5. Inference and Trustworthiness

diff-diff's `SyntheticDiD` supports two standard error methods:

- **Placebo SE** (default): permutes which control units are pretended to be "treated" and re-estimates SDiD on each permutation. The standard deviation of those placebo effects is the SE. Matches the algorithm in Arkhangelsky et al. (2021), Algorithm 4.
- **Bootstrap SE**: resamples units (with replacement), re-estimates the unit weights, and uses the resulting distribution. Matches the R `synthdid` bootstrap variant.

Both methods are configured on the `SyntheticDiD` constructor, not on `.fit()`. Use placebo by default (it's the published method); switch to bootstrap if you want to cross-check.

In [ ]:
# Refit with bootstrap SE for comparison. Same n_bootstrap and seed.
model_boot = SyntheticDiD(variance_method="bootstrap", n_bootstrap=100, seed=42)
results_boot = model_boot.fit(
    df,
    outcome="conversions",
    treatment="treated",
    unit="market_id",
    time="week",
    post_periods=[7, 8, 9, 10, 11, 12],
)

comparison = pd.DataFrame({
    "Method": ["Placebo (default)", "Bootstrap"],
    "ATT": [results.att, results_boot.att],
    "SE": [results.se, results_boot.se],
    "CI_low": [results.conf_int[0], results_boot.conf_int[0]],
    "CI_high": [results.conf_int[1], results_boot.conf_int[1]],
})
comparison.round(2)

In [ ]:
# Automated diagnostic checklist following Baker et al. (2025).
# practitioner_next_steps prints a formatted summary AND returns a dict;
# assigning to `_` keeps Jupyter from re-displaying the dict.
_ = practitioner_next_steps(results)

**Why no parallel-trends test?** The Baker et al. workflow includes an explicit parallel-trends check (Step 3), but `practitioner_next_steps` skips it for `SyntheticDiD`. That's intentional: SDiD relaxes the parallel-trends assumption *by construction* - the unit and time weights are chosen to make the synthetic control's pre-trend match the treated pre-trend. The parallel-trends check is replaced by the pre-treatment fit diagnostic in Step 6, which we already inspected (RMSE well below the noise level).

## 6. Cross-validation on GeoLift's Public Data

If you've used [GeoLift](https://github.com/facebookincubator/GeoLift) (Meta's open-source geo-experimentation R package), you know its Simulated Retail dataset. It's a 40-city panel of 105 daily observations - the canonical example in the GeoLift Walkthrough vignette uses Chicago and Portland as test markets and days 91-105 as the treatment window.

We bundle the same dataset (extracted from `GeoLift_Test.rda`, MIT licensed) at `docs/tutorials/data/geolift_test.csv`. Here we run diff-diff's SyntheticDiD against it and compare to GeoLift's published Augmented Synthetic Control (ASCM) numbers.

**Honesty note**: GeoLift uses ASCM (Ben-Michael, Feller & Rothstein 2021), not SDiD. The two methods are related but not identical, so we expect different point estimates. The point of this section is "cross-library cross-validation," not exact replication.

In [ ]:
geolift = pd.read_csv("data/geolift_test.csv")
print(f"Shape: {geolift.shape}")
print(f"Locations: {len(set(geolift['location']))}")
print(f"Days: {geolift['day'].min()}-{geolift['day'].max()}")
print(f"Treated: {sorted(set(geolift.loc[geolift.treated == 1, 'location']))}")
geolift.head()

In [ ]:
# Same SDiD configuration as Section 3
geolift_model = SyntheticDiD(variance_method="placebo", n_bootstrap=100, seed=42)
geolift_results = geolift_model.fit(
    geolift,
    outcome="Y",
    treatment="treated",
    unit="location",
    time="day",
    post_periods=list(range(91, 106)),  # days 91-105 are post (15 days)
)

In [ ]:
# GeoLift Walkthrough vignette canonical example output
# Source: https://github.com/facebookincubator/GeoLift/blob/main/vignettes/GeoLift_Walkthrough.md
# Method: Augmented Synthetic Control (ASCM)
GEOLIFT_PUBLISHED_ATT = 155.556     # daily average across 15-day post period
GEOLIFT_PUBLISHED_LIFT_PCT = 5.4

treated_pre_mean = float(geolift[(geolift.treated == 1) & (geolift.day < 91)]["Y"].mean())
sdid_lift_pct = 100 * geolift_results.att / treated_pre_mean

comparison = pd.DataFrame({
    "Method": ["GeoLift (ASCM, published)", "diff-diff SyntheticDiD"],
    "ATT": [GEOLIFT_PUBLISHED_ATT, geolift_results.att],
    "Lift %": [GEOLIFT_PUBLISHED_LIFT_PCT, sdid_lift_pct],
})
print(comparison.round(2).to_string(index=False))

print(f"\ndiff-diff also reports:")
print(f"  SE:           {geolift_results.se:.2f}")
print(f"  95% CI:       ({geolift_results.conf_int[0]:.2f}, {geolift_results.conf_int[1]:.2f})")
print(f"  Pre-fit RMSE: {geolift_results.pre_treatment_fit:.2f}")

**Reading these numbers honestly.** Both methods agree the campaign in Chicago and Portland produced a positive lift in the 5-10% range. The point estimates differ - GeoLift's ASCM reports 155.56 (5.4% lift), diff-diff's SDiD reports about 238 (8.2% lift). The methods differ in regularization and bias correction, so some divergence is expected on a real-world dataset like this.

Notice diff-diff's 95% CI crosses zero. SDiD is also flagging a "poor pre-fit" warning (pre-treatment RMSE exceeds the standard deviation of treated outcomes in the pre-period). With only **2 treated cities** and 38 controls - none of which look quite like Chicago or Portland - the synthetic match is loose, and SDiD propagates that uncertainty into the standard error.

This is a feature, not a failure. SDiD is telling you when the data doesn't support a confident claim. A different method might report a tighter interval on the same data, but you should ask why before believing it.

For everyday practice, if your geo-experiment has 15+ treated markets and a clean pre-trend (like the synthetic walkthrough in Sections 2-5), SDiD will give you tight confidence intervals you can take to leadership. When the data is noisier, SDiD will tell you so.

## 7. Communicating Results to Leadership

A stakeholder-ready summary of the synthetic walkthrough (Section 3-5 results):

> **Headline.** The campaign lifted weekly conversions in test markets by approximately **297 per market per week** (95% CI: 263 to 330), or about an **18% lift** on a baseline of ~1,600 conversions per market per week.
>
> **Sample size and design.** 18 test markets, 62 control markets. 12 weeks of weekly data: 6 weeks pre-launch, 6 weeks post-launch. Outcome: weekly conversions per market.
>
> **Validity evidence.** The synthetic control's pre-treatment fit was tight (RMSE 64 on a noise level of ~137 - the synthetic blend tracks the treated pre-trend closely). The placebo standard error matches the published Arkhangelsky et al. (2021) method. The estimate is statistically significant at the p < 0.01 level. Pre-period gaps between treated markets and the synthetic control are within ±20 conversions per week, consistent with normal noise.
>
> **What "297 conversions per market per week" means in business terms.** Across 18 treated markets and 6 weeks, that's roughly 32,000 incremental conversions attributable to the campaign. Translate to your own revenue-per-conversion to compare against campaign spend.
>
> **Practical significance caveat.** An 18% lift is large by marketing benchmarks, but whether it justifies the campaign cost is a business judgment, not a statistical one. The statistics tell you the lift exists; the business case tells you whether the lift was worth paying for.

Adapt this template for your own campaign by swapping in your numbers from `results.summary()`, your own market counts, your own pre-period diagnostics, and your own conversion-to-revenue translation. The pattern - **headline → sample size → validity evidence → business interpretation → practical significance** - is the part to keep.

## 8. What diff-diff's SyntheticDiD Adds for the GeoLift/CausalImpact Audience

This tutorial taught one estimator on one design pattern (block treatment, simultaneous launch in a subset of geos). For that pattern, here's what reaching for diff-diff gets you:

- **Canonical SDiD implementation.** diff-diff's `SyntheticDiD` is the [Arkhangelsky et al. (2021)](https://www.aeaweb.org/articles?id=10.1257/aer.20190159) method as published - both unit AND time weights, the placebo SE method from the paper, and a regularization scheme matching the R `synthdid` reference. If your team has been hand-rolling synthetic control or using older library implementations, this is the upgrade path.
- **Full diagnostic interpretability.** `get_unit_weights_df()`, `get_time_weights_df()`, and `pre_treatment_fit` let you show stakeholders exactly which control markets are doing the work and how well the synthetic match held up. Section 4 of this tutorial walked through every diagnostic.
- **Survey weighting on the same estimator.** If you ever combine geo-experiment data with survey weights (e.g., aggregating brand-tracking responses to the market level), `SyntheticDiD` accepts a `SurveyDesign` argument with `pweight` support. See Tutorial 16 for the survey-design machinery.
- **Python-native and panel-first.** You keep unit-level observations and the rest of the diff-diff toolkit (parallel-trends checks, placebo tests, dedicated tutorials for other DiD designs) available in the same library when your next analysis needs something different.

For a side-by-side feature comparison against GeoLift and CausalImpact (including scenarios outside this tutorial's scope), see the planned `B2c` comparison page in [`ROADMAP.md`](https://github.com/igerber/diff-diff/blob/main/ROADMAP.md). Until that ships, the practitioner-facing entry points are [`practitioner_getting_started.rst`](practitioner_getting_started.html) and [`practitioner_decision_tree.rst`](practitioner_decision_tree.html).